# Taazi Khabar — QLoRA Fine-Tuning

Fine-tune a single LoRA adapter on all 3 UPSC tasks (summarizer, article filter, quiz setter).
The base model learns to switch behavior based on the `instruction` field in each example.

**Usage:**
1. Upload `combined.jsonl` (from `app/ai/training/data/processed/`) to Google Drive.
2. Set `DATASET_PATH` below.
3. Run all cells.

**Tasks in the dataset:**
- `Summarize this UPSC article...` → structured GK summary (298 examples)
- `Determine if this is UPSC-relevant...` → YES/NO (311 examples)
- `Generate UPSC-style MCQ...` → MCQ with 4 options (95 examples)

In [ ]:
# --- Configuration ---
# Point this at your combined JSONL file (Alpaca format)
DATASET_PATH = "/content/drive/MyDrive/TaaziKhabar/training/combined.jsonl"

# Model to fine-tune (pick one that fits your GPU)
# Colab T4 (16GB) can handle ~3-7B models with QLoRA
BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"  # 3.8B, good for Colab
# BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"  # 7B, needs more memory

# LoRA hyperparameters
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
MAX_SEQ_LENGTH = 2048

# Output
OUTPUT_DIR = "/content/drive/MyDrive/TaaziKhabar/adapters/combined-v1"
HF_TOKEN = None  # Set if using gated models like Llama

## 1. Install Dependencies

In [ ]:
!pip install -qU \
    transformers \
    datasets \
    accelerate \
    peft \
    bitsandbytes \
    trl \
    wandb

## 2. Mount Google Drive (optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Load Dataset

In [ ]:
import json
from datasets import Dataset

records = []
with open(DATASET_PATH, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records)} records")
print("Sample:", json.dumps(records[0], indent=2)[:500])

## 4. Format into Alpaca-style Prompt

In [ ]:
def format_alpaca(record):
    """Format an Alpaca record into a chat-style prompt."""
    instruction = record.get("instruction", "")
    inp = record.get("input", "")
    output = record.get("output", "")

    if inp:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Response:\n{output}"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"
    return prompt


formatted = [format_alpaca(r) for r in records]
dataset = Dataset.from_list([{"text": t} for t in formatted])
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(dataset['train'])}, Eval: {len(dataset['test'])}")

## 5. Load Base Model with QLoRA

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Train

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
)

trainer.train()

## 7. Save Adapter

In [ ]:
adapter_path = f"{OUTPUT_DIR}/final"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")
print("Files:", os.listdir(adapter_path))

## 8. Inference Test

In [ ]:
from peft import PeftModel

# Reload base + adapter
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN,
)
model = PeftModel.from_pretrained(base, adapter_path)

test_prompt = "### Instruction:\nSummarize this UPSC article for GK preparation.\n\n### Input:\nThe Supreme Court has upheld the constitutional validity of the Places of Worship (Special Provisions) Act, 1991, ruling that it serves as a cornerstone of secularism.\n\n### Response:\n"

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.2,
    do_sample=True,
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))